# Oppitunti 13 - Agentin muisti


## Asennus

Tämä muistikirja näyttää, miten rakennetaan matkavarauksen agentti, jolla on **pysyvä muisti** käyttämällä **Microsoft Agent Frameworkia** (MAF).

Opit, miten erilaiset agentin muistityypit — työmuisti, lyhytaikainen muisti ja pitkäaikainen muisti — vaikuttavat siihen, miten agentti säilyttää ja käyttää tietoa keskustelujen välillä.

**Edellytykset:**
- Microsoft Foundry -projekti, johon on otettu käyttöön chat-malli (esim. `gpt-4.1-mini`).
- Kirjautuneena Azure CLI:llä — suorita `az login` päätelaitteessasi.
- `AZURE_AI_PROJECT_ENDPOINT` — Microsoft Foundry -projektisi päätepiste.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — käyttöön otetun mallisi nimi.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Agentin muistityypit

Tekoälyagentit voivat hyödyntää erilaisia muistin tyyppejä, joista kukin palvelee eri tarkoitusta:

### Työmuisti
Itse keskusteluketju — viestit, jotka vaihdetaan yhdessä istunnossa. Agentti voi viitata aiempiin viesteihin samassa ketjussa säilyttääkseen johdonmukaisuuden. MAFissa tämä luodaan **`agent.create_session()`** -metodilla, joka palauttaa `AgentSession`-olion.

### Lyhytaikainen muisti
Tiedot, jotka säilyvät tehtävän tai istunnon ajan, mutta joita ei tallenneta pysyvästi. Esimerkiksi agentti voi koota faktoja monimutkaisen suunnittelukeskustelun aikana ja käyttää niitä lopullisen aikataulun laatimiseen.

### Pitkäaikainen muisti
Mieltymykset ja faktat, jotka säilyvät **istuntojen yli**. Palaavan käyttäjän ei pitäisi joutua toistamaan ruokavaliörajoituksiaan tai matkustustyyliään. Pitkäaikainen muisti perustuu tyypillisesti ulkoiseen tallennustilaan — tietokantaan, tiedostoon tai vektori-indeksiin — ja se tuodaan agentin käyttöön työkalujen kautta.


## Työmuisti istuntojen kanssa

Muistamisen yksinkertaisin muoto on keskusteluistunto. Kun annat saman istunnon (luotu `agent.create_session()` -komennolla) peräkkäisiin `agent.run()` -kutsuihin, agentti näkee koko kyseisen keskustelun historian ja voi palauttaa mieleensä aiemmin sanottuja yksityiskohtia.

Luodaan matkatoimistoagentti ja havainnollistetaan työmuistia.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agentti muisti budjetin oikein, koska molemmat viestit kuuluvat samaan istuntoon. Tämä on **työmuisti** — se on olemassa vain istunnon elinkaaren ajan.

### Mitä tapahtuu uudessa ketjussa?

Jos luomme **uuden** istunnon, agentilla ei ole muistia edellisestä keskustelusta:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pitkän aikavälin muistikuviot

Muistaaksemme käyttäjän mieltymykset **istuntojen välillä**, tarvitsemme pysyvän tietovaraston, joka elää keskusteluketjun ulkopuolella. Agentti pääsee tähän varastoon käsiksi **työkalujen** kautta — funktioiden, joita se voi kutsua tallentaakseen ja hakiakseen tietoa.

Alla toteutamme yksinkertaisen muistissa olevan mieltymysten varaston (tuotantokäytössä tämän taustalla olisi tietokanta tai vektori-indeksi) ja tarjoamme sen työkaluina, joita agentti voi käyttää.

### Arkkitehtuuri
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Tapaus 1 — Uusi käyttäjä varaa juhlapäivämatkan

Sarah vierailee ensimmäistä kertaa. Agentin tulisi tallentaa hänen mieltymyksensä työkalujen avulla ja käyttää niitä hotellien suosittelemiseen.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Skenaario 2 — Sarah palaa viikkoja myöhemmin

Sarah aloittaa **täysin uuden keskustelun** (simuloiden uutta istuntoa). Työmuisti on tyhjä, mutta pitkäaikainen mieltymyspalvelin sisältää yhä hänen tietonsa. Agentin tulisi hakea ne ja käyttää niitä suositusten personointiin.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Yhteenveto

Tässä oppitunnissa opit kolme agentin muistityyppiä ja miten ne toteutetaan Microsoft Agent Frameworkin avulla:

| Muistityyppi | MAF-mekanismi | Kesto |
|---|---|---|
| **Työmuisti** | `agent.create_session()` | Yksi keskustelu |
| **Lyhytaikainen** | Kertynyt konteksti ketjussa | Yksi tehtävä / istunto |
| **Pitkäaikainen** | Ulkoinen tallennus @tool-funktioiden kautta | Istuntojen yli |

### Keskeiset Opit
1. **`agent.create_session()`** tarjoaa työmuistin — agentti näkee koko keskusteluhistorian istunnon aikana.
2. **Uudet istunnot menettävät kontekstin** — ilman pitkäaikaista muistia agentti ei muista aiempia keskusteluja.
3. **`@tool`-funktiot** yhdistävät aukon — ne antavat agentille mahdollisuuden tallentaa ja hakea tietoa pysyvästä tietovarastosta.
4. **Personalisointi paranee ajan myötä** — mitä enemmän mieltymyksiä tallennetaan, sitä parempia suosituksia agentti antaa.

### Käytännön Sovelluksia
- **Asiakaspalvelu**: Muistaa asiakkaan historian ja mieltymykset
- **Henkilökohtaiset avustajat**: Säilyttää kontekstin päivien tai viikkojen ajan
- **Terveydenhuolto**: Seuraa potilastietoja ja mieltymyksiä
- **Verkkokauppa**: Personoitu ostokokemus historian perusteella

### Seuraavat Askeleet
- Korvaa muistissa oleva sanakirja tietokannalla tai vektorivarastolla (esim. Azure AI Search)
- Lisää muistin vanhentuminen aikaherkille tiedoille
- Rakenna moniagenttijärjestelmiä jaetulla muistilla
- Tutustu Cognee-muistikirjaan, joka tukee tietämyskaavioita


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
